# Forensic Diagnosis: Single-Bin AD Pipeline (E = 1.014 MeV by default)

This notebook **replays the production EXFOR-to-ENDF angular-distribution pipeline at one energy bin**
using exactly the same library functions as `scripts/exfor_to_endf_sampling_v2.py`. It is meant for
forensic investigation of a single bin, not for production use.

## How to run on another bin

Set `TARGET_ENERGY_MEV` in the **Config** cell, then *Restart and Run All*. Everything below adapts
automatically.

## What we are testing (initial focus: E = 1.014 MeV)

In the production run `new_test_50/`, the published nominal at 1.014 MeV is unphysical: wrong slope at
both extremes, and **dσ/dΩ goes negative at backward angles** even with `APPLY_POSITIVITY_PROJECTION=True`.
Cierjacks (4 pts) appears to dominate over Kinney (8 pts).

Four hypotheses we want this notebook to confirm or refute:

1. **H1 — Positivity projection is applied only to MC samples, not to the nominal.**
   `project_to_positive_distribution` is called inside the per-sample loop at
   `exfor_to_endf_sampling_v2.py:642` and `:683`; nothing checks the nominal.
2. **H2 — `L=5` Legendre fit on 12 points (`N_eff=2.8`) rings.** AICc picks the highest tested degree
   because residuals reward extra flexibility, but the curve has no constraint outside the data.
3. **H3 — Asymmetric uncertainty floor on Cierjacks.** Backward Cierjacks points get inflated from
   ~0.1% to ~9.1%; forward points keep their ~7%. Same experiment, two different effective shapes.
4. **H4 — GLS-ESS shrinks per-point weight as `1/(1+(n_j−1)ρ)`.** Kinney has n=8 → larger shrink;
   Cierjacks has n=4 → smaller shrink. Combined with H3, Cierjacks ends with higher *summed* weight.

Each diagnostic cell ends with the explicit hypothesis it tests.

In [ ]:
# --- Imports & path setup ---------------------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.polynomial.legendre import legval, legvander

# Make the kika repo importable (this notebook lives in kika/exfor/tests/)
_kika_repo_root = Path(__file__).resolve().parent.parent.parent.parent if "__file__" in globals() else Path.cwd().resolve()
while _kika_repo_root != _kika_repo_root.parent:
    if (_kika_repo_root / "scripts" / "exfor_to_endf_sampling_v2.py").exists():
        break
    _kika_repo_root = _kika_repo_root.parent
if str(_kika_repo_root) not in sys.path:
    sys.path.insert(0, str(_kika_repo_root))

# kika modules
from kika.exfor import read_all_exfor
import kika.exfor as exfor_module

# Pipeline modules (production helpers we reuse — DO NOT reimplement)
from scripts.exfor_utils import (
    EnergyBinInfo,
    build_exfor_cache_from_objects,
    build_union_energy_grid,
    compute_energy_bins_with_tof_resolution,
    filter_exfor_with_energy_bin,
    apply_uncertainty_floor,
    apply_per_experiment_weight_cap,
)
from scripts.resample_AD import (
    sample_legendre_coefficients,
    endf_normalize_legendre_coeffs,
    check_angular_distribution_positivity,
    project_to_positive_distribution,
)

print("Imports OK")
print(f"kika repo root: {_kika_repo_root}")

In [ ]:
# --- Config (mirrors scripts/exfor_to_endf_sampling_v2.py:169-279) ----------
# Change THIS to investigate a different bin and Restart-and-Run-All.
TARGET_ENERGY_MEV = 1.014

# Paths / I/O
ENDF_FILE = "/share_snc/snc/JuanMonleon/jeff40_with_MF4_from_jeff33/26-Fe-56g.txt"
EXFOR_DB_PATH = "/share_snc/snc/JuanMonleon/EXFOR/x4_iron_angular.db"
SUPPLEMENTARY_JSON_FILES = [
    "/share_snc/snc/JuanMonleon/EXFOR/data_v1/27673002.json",
]

# Data source / target
TARGET_ZAIDS = [26056, 26000]
TARGET_PROJECTILE = "N"
MT_NUMBER = 2

# Energy range / physics
ENERGY_MIN_MEV = 0.847
ENERGY_MAX_MEV = 4.0
M_PROJ_U = 1.008665
M_TARG_U = 55.93494

# Legendre fitting
MAX_LEGENDRE_DEGREE = 6
SELECT_DEGREE = "aicc"
RIDGE_LAMBDA = 1e-4
RIDGE_POWER = 4
DF_METHOD = "hat"

# Uncertainty / discrepancy
USE_BAND_DISCREPANCY = True
MIN_POINTS_PER_BAND = 5
MAX_BAND_SCALE_FACTOR = 5.0
NORMALIZATION_SIGMA = 0.05
EXCLUDE_EXPERIMENTS = ["32246002"]
MIN_RELATIVE_UNCERTAINTY = 0.01
UNCERTAINTY_FLOOR_STRATEGY = "bin_median"
RESCALE_UNC_BY_CHI2 = True
ALLOW_SHRINK_UNC = True

# Energy binning / kernel weighting
NORMALIZE_BY_N_POINTS = True
MAX_EXP_WEIGHT_FRAC_BIN = 0.80
ENERGY_GRID_SOURCE = "union"
UNION_GRID_SUBENTRIES = [("10571002", 0.847, 2.5), ("23365005", 2.5, 4.0)]
DELTA_T_NS = 5.0
FLIGHT_PATH_M = 27.037
N_SIGMA_CUTOFF = 3.0

# Positivity / sampling
APPLY_POSITIVITY_PROJECTION = True
POSITIVITY_CHECK_POINTS = 101
BASE_SEED = 42

# Configure EXFOR backend
exfor_module.configure(db_path=EXFOR_DB_PATH)

print(f"TARGET_ENERGY_MEV = {TARGET_ENERGY_MEV}")

In [ ]:
# --- Load EXFOR + build union grid + locate the bin -------------------------
print("Loading EXFOR ...")
exfor_dict, _load_status = read_all_exfor(
    target=TARGET_ZAIDS,
    mt=MT_NUMBER,
    source="database",
    group_by_energy=False,
    supplementary_json_files=SUPPLEMENTARY_JSON_FILES,
    exclude_experiments=EXCLUDE_EXPERIMENTS,
    return_load_status=True,
)
exfor_cache, sorted_exfor_energies = build_exfor_cache_from_objects(
    list(exfor_dict.values()),
    exclude_experiments=EXCLUDE_EXPERIMENTS,
)
print(f"  loaded {len(exfor_dict)} datasets, {len(sorted_exfor_energies)} unique energies")

# Build the same union grid the production pipeline uses (no ENDF dependence
# for the bin boundaries — production uses ENERGY_GRID_SOURCE='union').
grid_energies_ev = build_union_energy_grid(
    exfor_cache=exfor_cache,
    subentries=UNION_GRID_SUBENTRIES,
    energy_min_mev=ENERGY_MIN_MEV,
    energy_max_mev=ENERGY_MAX_MEV,
)
energy_bins = compute_energy_bins_with_tof_resolution(
    energies_ev=grid_energies_ev,
    energy_min_mev=ENERGY_MIN_MEV,
    energy_max_mev=ENERGY_MAX_MEV,
    delta_t_ns=DELTA_T_NS,
    flight_path_m=FLIGHT_PATH_M,
)
print(f"  union grid: {len(grid_energies_ev)} pts -> {len(energy_bins)} bins")

# Find bin closest to TARGET_ENERGY_MEV
distances = np.array([abs(b.energy_mev - TARGET_ENERGY_MEV) for b in energy_bins])
bin_idx = int(np.argmin(distances))
bin_info = energy_bins[bin_idx]
print(f"\nClosest bin to {TARGET_ENERGY_MEV} MeV:")
print(f"  index={bin_info.index}  E={bin_info.energy_mev:.4f} MeV")
print(f"  bin range = [{bin_info.bin_lower_mev:.4f}, {bin_info.bin_upper_mev:.4f}] MeV "
      f"(width {(bin_info.bin_upper_mev - bin_info.bin_lower_mev)*1e3:.2f} keV)")
print(f"  sigma_E (TOF) = {bin_info.sigma_E_mev*1e3:.2f} keV")

In [ ]:
# --- Run the production filter + Plot 1: data scatter -----------------------
exfor_df, experiments_info, kernel_weights, diagnostics, floor_stats = filter_exfor_with_energy_bin(
    exfor_cache=exfor_cache,
    sorted_energies=sorted_exfor_energies,
    bin_lower_mev=bin_info.bin_lower_mev,
    bin_upper_mev=bin_info.bin_upper_mev,
    target_energy_mev=bin_info.energy_mev,
    m_proj_u=M_PROJ_U,
    m_targ_u=M_TARG_U,
    dedupe_per_experiment=True,
    exclude_experiments=EXCLUDE_EXPERIMENTS,
    min_relative_uncertainty=MIN_RELATIVE_UNCERTAINTY,
    unc_floor_strategy=UNCERTAINTY_FLOOR_STRATEGY,
    normalize_by_n_points=NORMALIZE_BY_N_POINTS,
    sigma_norm=NORMALIZATION_SIGMA,
    max_experiment_weight_fraction=MAX_EXP_WEIGHT_FRAC_BIN,
)

if "experiment_id" not in exfor_df.columns:
    exfor_df["experiment_id"] = exfor_df["entry"].astype(str) + "/" + exfor_df["subentry"].astype(str)

# Per-experiment summary table (production weight totals).
exp_summary = pd.DataFrame([
    {
        "experiment": f"{e.get('entry')}/{e.get('subentry')}",
        "author": e.get("author", "?"),
        "year": e.get("year", "?"),
        "n_points": e.get("n_points", 0),
        "exfor_E_MeV": e.get("exfor_energy_mev", np.nan),
    }
    for e in experiments_info
])

# Sum kernel weights per experiment to confirm the 0.319 / 0.596 numbers in the log.
total_w = float(np.sum(kernel_weights)) if len(kernel_weights) else 1.0
weight_sum_by_exp = (
    pd.DataFrame({"experiment_id": exfor_df["experiment_id"].values, "w": kernel_weights})
    .groupby("experiment_id")["w"].sum()
    / total_w
)
exp_summary["weight_frac"] = exp_summary["experiment"].map(weight_sum_by_exp).fillna(0.0)

print(f"N_points = {len(exfor_df)}, N_experiments = {len(experiments_info)}, "
      f"N_eff = {diagnostics.n_eff:.2f}")
print(f"floor_stats: n_floored={floor_stats['n_floored']}, "
      f"n_total={floor_stats.get('n_total', 'n/a')}, "
      f"replacement_rel={floor_stats.get('replacement_rel', np.nan):.4f}")
print()
print(exp_summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
markers = ["o", "s", "^", "v", "D", "P", "*", "X"]
for i, exp_id in enumerate(exfor_df["experiment_id"].unique()):
    sub = exfor_df[exfor_df["experiment_id"] == exp_id]
    label = exp_id
    e_info = next((e for e in experiments_info
                   if f"{e.get('entry')}/{e.get('subentry')}" == exp_id), {})
    if e_info:
        label = f"{e_info.get('author','?')} ({e_info.get('year','?')}) — {exp_id}"
    ax.errorbar(sub["mu"], sub["value"], yerr=sub["unc"],
                fmt=markers[i % len(markers)], capsize=3, alpha=0.85, label=label)
ax.axvspan(-1, -0.5, alpha=0.07, color="red", label="backward band (μ<−0.5)")
ax.axvspan(0.5, 1, alpha=0.07, color="blue", label="forward band (μ>0.5)")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel(r"$\mu = \cos\theta_{CM}$")
ax.set_ylabel(r"$d\sigma/d\Omega$ (b/sr)")
ax.set_yscale("log")
ax.set_title(f"EXFOR data @ E = {bin_info.energy_mev:.4f} MeV "
             f"(bin [{bin_info.bin_lower_mev:.4f}, {bin_info.bin_upper_mev:.4f}] MeV)")
ax.legend(fontsize=8, loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# NB: the production log line "10571.002 ... w=0.319" is the PER-POINT kernel weight
# (each Kinney point), not the per-experiment summed fraction. We will see that
# 0.319 vs 0.596 reappears as `w_kernel_prod` in Cell 6 (the per-point breakdown),
# while `weight_frac` here is sum_per_exp / total — a different metric.
print("\n[Q] Per-experiment summed weight_frac is shown above. The log's w=0.319 / w=0.596 "
      "values are PER-POINT kernel weights — Cell 6 will reproduce them in the w_kernel_prod "
      "column. Two complementary views: (i) per-point weight (Cierjacks heavier), (ii) summed "
      "weight (Kinney slightly heavier because it has more points).")

In [ ]:
# --- H3: Per-point uncertainty floor breakdown -------------------------------
# Re-run filter_exfor_with_energy_bin with min_relative_uncertainty=0 to get the
# RAW (pre-floor) `unc` values, then compare to the post-floor `exfor_df` already
# loaded. Joining on (entry, subentry, value) since value is preserved through
# LAB→CM but mu is not.

raw_df, _, _, _, _ = filter_exfor_with_energy_bin(
    exfor_cache=exfor_cache,
    sorted_energies=sorted_exfor_energies,
    bin_lower_mev=bin_info.bin_lower_mev,
    bin_upper_mev=bin_info.bin_upper_mev,
    target_energy_mev=bin_info.energy_mev,
    m_proj_u=M_PROJ_U, m_targ_u=M_TARG_U,
    dedupe_per_experiment=True,
    exclude_experiments=EXCLUDE_EXPERIMENTS,
    min_relative_uncertainty=0.0,  # NO floor
    unc_floor_strategy=UNCERTAINTY_FLOOR_STRATEGY,
    normalize_by_n_points=NORMALIZE_BY_N_POINTS,
    sigma_norm=NORMALIZATION_SIGMA,
    max_experiment_weight_fraction=MAX_EXP_WEIGHT_FRAC_BIN,
)
if "experiment_id" not in raw_df.columns:
    raw_df["experiment_id"] = raw_df["entry"].astype(str) + "/" + raw_df["subentry"].astype(str)

# Join post-floor frame onto pre-floor on (entry, subentry, value).
join_cols = ["entry", "subentry", "value"]
joined = raw_df.merge(
    exfor_df[[*join_cols, "mu", "unc"]].rename(columns={"unc": "unc_floored"}),
    on=join_cols, suffixes=("_raw", ""),
)

cmp = pd.DataFrame({
    "experiment": joined["experiment_id"].values,
    "mu": joined["mu"].values,
    "value": joined["value"].values,
    "unc_raw": joined["unc"].values,
    "rel_raw": joined["unc"].values / np.abs(joined["value"].values),
    "unc_floored": joined["unc_floored"].values,
    "rel_floored": joined["unc_floored"].values / np.abs(joined["value"].values),
})
cmp["was_floored"] = (cmp["unc_raw"] - cmp["unc_floored"]).abs() > 1e-12
cmp["band"] = pd.cut(cmp["mu"], bins=[-1.001, -0.5, 0.5, 1.001], labels=["B", "M", "F"])
cmp = cmp.sort_values(["experiment", "mu"]).reset_index(drop=True)
floor_stats_redux = {"replacement_rel": float("nan"), "n_floored": int(cmp["was_floored"].sum())}

print(f"Floor strategy: {UNCERTAINTY_FLOOR_STRATEGY}, threshold: {MIN_RELATIVE_UNCERTAINTY*100:.1f}%, "
      f"replacement_rel: {floor_stats_redux.get('replacement_rel', np.nan):.4f}")
print(f"Total points: {len(cmp)}, n_floored: {int(cmp['was_floored'].sum())}")
print()
# Per-experiment × band summary
per_exp_band = (
    cmp.groupby(["experiment", "band"], observed=False)
    .agg(n=("value", "size"),
         rel_raw_mean=("rel_raw", "mean"),
         rel_floored_mean=("rel_floored", "mean"),
         n_floored=("was_floored", "sum"))
)
print("Per-experiment × band:")
print(per_exp_band.to_string())
print()
print("Per-point detail:")
print(cmp.to_string(index=False))

print("\n[Q-H3] Are 3/3 Cierjacks BACKWARD points (μ<−0.5) flagged was_floored=True with "
      "rel_floored close to the bin median, while Cierjacks FORWARD points have a different "
      "rel_raw and may or may not be floored?")

In [ ]:
# --- H4: GLS-ESS per-point weight decomposition ------------------------------
# Reconstruct the per-point weight chain manually so we can see each stage:
#   w_stat   = 1 / sigma^2
#   w_norm   = w_stat * (1 / (1 + (n_j-1) * rho_j))    # GLS-ESS shrink (per experiment)
#       where rho_j = sigma_norm^2 / (sigma_norm^2 + sigma_rel_j^2)
#   w_capped = apply_per_experiment_weight_cap(...)    # global 80% cap
# This mirrors the production logic in filter_exfor_with_energy_bin.

g_stat = 1.0 / np.maximum(exfor_df["unc"].values, 1e-18) ** 2

# ρ per experiment using post-floor mean rel unc, n_j = total points per experiment.
n_per_exp = exfor_df.groupby("experiment_id")["mu"].transform("size").values
rel_unc = exfor_df["unc"].values / np.abs(exfor_df["value"].values)
sn2 = NORMALIZATION_SIGMA ** 2
rho = sn2 / (sn2 + rel_unc ** 2)
shrink = 1.0 / (1.0 + np.maximum(n_per_exp - 1, 0) * rho) if NORMALIZE_BY_N_POINTS else np.ones_like(rho)
g_norm = g_stat * shrink

# Cap stage — re-run apply_per_experiment_weight_cap to get the post-cap weights.
w_capped, cap_diag, cap_applied = apply_per_experiment_weight_cap(
    exfor_df, kernel_weights, max_experiment_weight_fraction=MAX_EXP_WEIGHT_FRAC_BIN,
)

per_point = pd.DataFrame({
    "experiment_id": exfor_df["experiment_id"].values,
    "mu": exfor_df["mu"].values,
    "rel_unc": rel_unc,
    "n_per_exp": n_per_exp,
    "rho": rho,
    "shrink_factor": shrink,
    "w_stat": g_stat,
    "w_after_norm": g_norm,
    "w_kernel_prod": kernel_weights,    # what the filter actually returned
    "w_after_cap": w_capped,
})
print("Per-point weight stages (subset):")
print(per_point.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print()
print(f"cap_applied: {cap_applied}, cap_diag: {cap_diag}")

# Aggregated per-experiment fractions across all stages
def _frac(arr):
    s = float(arr.sum())
    return arr / s if s > 0 else arr * 0.0

agg = pd.DataFrame({
    "experiment_id": exfor_df["experiment_id"].values,
    "stat": _frac(g_stat),
    "after_GLS_ESS": _frac(g_norm),
    "kernel_weights (prod return)": _frac(kernel_weights),
    "after_cap": _frac(w_capped),
}).groupby("experiment_id").sum()
print("\nFraction of total weight per experiment, at each pipeline stage:")
print(agg.to_string(float_format=lambda x: f"{x:.3f}"))

fig, ax = plt.subplots(figsize=(9, 5))
agg.plot(kind="bar", ax=ax)
ax.set_ylabel("fraction of total weight")
ax.set_title("Per-experiment weight fraction at each pipeline stage")
ax.axhline(MAX_EXP_WEIGHT_FRAC_BIN, ls="--", color="red",
           label=f"MAX_EXP_WEIGHT_FRAC_BIN = {MAX_EXP_WEIGHT_FRAC_BIN}")
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print("\n[Q-H4] At E=1.014 MeV: does Cierjacks's weight_frac drop ONLY a little going from "
      "'stat' to 'after_GLS_ESS' (n=4, smaller shrink), while Kinney drops more (n=8)? "
      "Does the global 80% cap leave the imbalance untouched?")

In [ ]:
# --- H1+H2: Nominal Legendre fit on the bin ----------------------------------
coef_df, fit_info = sample_legendre_coefficients(
    exfor_df,
    value_col="value",
    unc_col="unc",
    degree=None,
    max_degree=MAX_LEGENDRE_DEGREE,
    select_degree=SELECT_DEGREE,
    ridge_lambda=RIDGE_LAMBDA,
    ridge_power=RIDGE_POWER,
    df_method=DF_METHOD,
    external_weights=kernel_weights,
    n_samples=1,
    rescale_unc_by_chi2=RESCALE_UNC_BY_CHI2,
    allow_shrink_unc=ALLOW_SHRINK_UNC,
    use_band_discrepancy=USE_BAND_DISCREPANCY,
    min_points_per_band=MIN_POINTS_PER_BAND,
    max_band_scale=MAX_BAND_SCALE_FACTOR,
)
nominal_coeffs = coef_df.iloc[0].to_numpy()
fitted_degree = fit_info["degree"]
chi2_red = fit_info["chi2_red"]
all_degrees_info = fit_info.get("all_degrees_info") or {}
tau_info = fit_info.get("tau_info", {})

print(f"Selected degree: L = {fitted_degree}")
print(f"chi2/dof = {chi2_red:.3f}")
print(f"N_eff (from filter) = {diagnostics.n_eff:.2f}")
print()
if all_degrees_info:
    aicc_table = pd.DataFrame({
        "L": list(all_degrees_info.keys()),
        "chi2_red": [all_degrees_info[d].get("chi2_red", np.nan) for d in all_degrees_info],
        "AICc": [all_degrees_info[d].get("aicc", np.nan) for d in all_degrees_info],
    })
    aicc_table["AICc-min(AICc)"] = aicc_table["AICc"] - aicc_table["AICc"].min()
    aicc_table["w_aicc"] = np.exp(-0.5 * aicc_table["AICc-min(AICc)"])
    aicc_table["w_aicc"] /= aicc_table["w_aicc"].sum()
    print("Per-degree fit & AICc weights:")
    print(aicc_table.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print()
print(f"tau_info: {tau_info}")
print(f"Nominal coeffs (c_0..c_L): {nominal_coeffs}")

# Dense plot — 1001 points on [-1, 1] (NOT 101!) so we catch between-grid negativity.
mu_dense = np.linspace(-1, 1, 1001)
y_dense = legval(mu_dense, nominal_coeffs)
min_p = float(np.min(y_dense))
print(f"min dσ/dΩ on dense 1001-pt grid: {min_p:+.4e}  (positive? {min_p >= 0})")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), height_ratios=[3, 1], sharex=True)
ax1.errorbar(exfor_df["mu"], exfor_df["value"], yerr=exfor_df["unc"],
             fmt="o", color="k", capsize=3, markersize=5, label="EXFOR (post-floor)")
ax1.plot(mu_dense, y_dense, "b-", lw=2,
         label=f"Nominal fit  L={fitted_degree}, χ²/dof={chi2_red:.2f}")
ax1.axhline(0, color="red", ls="--", lw=1)
ax1.axvspan(-1, -0.5, alpha=0.07, color="red")
ax1.axvspan(0.5, 1, alpha=0.07, color="blue")
ax1.set_ylabel(r"$d\sigma/d\Omega$ (b/sr)")
ax1.set_title(f"Nominal fit @ E = {bin_info.energy_mev:.4f} MeV   (min(p) = {min_p:+.3e})")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_yscale("symlog", linthresh=1e-3)

# residuals
y_at_data = legval(exfor_df["mu"].values, nominal_coeffs)
res = (exfor_df["value"].values - y_at_data) / exfor_df["unc"].values
ax2.scatter(exfor_df["mu"], res, c="k")
ax2.axhline(0, color="b", ls="--", lw=1)
ax2.axhline(2, color="red", ls=":", lw=1)
ax2.axhline(-2, color="red", ls=":", lw=1)
ax2.set_xlabel(r"$\mu$")
ax2.set_ylabel(r"$(y_\mathrm{exp}-y_\mathrm{fit})/\sigma$")
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n[Q-H1+H2] Does the nominal curve dip below zero anywhere on the dense grid? "
      "Is the chosen L = 5 with AICc weight ≥ 70%?")

In [ ]:
# --- H1: Apply positivity check + projector to the NOMINAL --------------------
# Production never does this for the nominal — see exfor_to_endf_sampling_v2.py:642/683.
# Here we run the same check_angular_distribution_positivity / project_to_positive_distribution
# functions on the nominal coeffs to show what the published curve WOULD look like
# if the projector were applied.

c0 = float(nominal_coeffs[0])
ok_prod_grid = check_angular_distribution_positivity(nominal_coeffs, POSITIVITY_CHECK_POINTS)
ok_dense = bool(np.min(legval(np.linspace(-1, 1, 1001), nominal_coeffs)) >= 0)
print(f"check_angular_distribution_positivity at {POSITIVITY_CHECK_POINTS} pts: {ok_prod_grid}")
print(f"min(p) on dense 1001-pt grid is non-negative: {ok_dense}")

# Project — freeze c_0 (and orders > MAX_LEGENDRE_DEGREE if any extra were padded).
frozen = {0: c0}
projected = project_to_positive_distribution(
    nominal_coeffs, n_points=POSITIVITY_CHECK_POINTS, frozen_indices=frozen,
)
print(f"\nProjected coeffs differ from original by: "
      f"max|Δ| = {np.max(np.abs(projected - nominal_coeffs)):.4e}")
print(f"min(p) on dense 1001-pt grid AFTER projection: "
      f"{float(np.min(legval(np.linspace(-1,1,1001), projected))):+.4e}")

mu_dense = np.linspace(-1, 1, 1001)
fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(exfor_df["mu"], exfor_df["value"], yerr=exfor_df["unc"],
            fmt="o", color="k", capsize=3, markersize=5, label="EXFOR (post-floor)")
ax.plot(mu_dense, legval(mu_dense, nominal_coeffs), "b-", lw=2,
        label=f"Production nominal  min(p)={np.min(legval(mu_dense, nominal_coeffs)):+.2e}")
ax.plot(mu_dense, legval(mu_dense, projected), "g--", lw=2,
        label=f"Hypothetical: nominal AFTER positivity projection  "
              f"min(p)={np.min(legval(mu_dense, projected)):+.2e}")
ax.axhline(0, color="red", ls="--", lw=1)
ax.axvspan(-1, -0.5, alpha=0.07, color="red")
ax.set_xlabel(r"$\mu$")
ax.set_ylabel(r"$d\sigma/d\Omega$ (b/sr)")
ax.set_title(f"Nominal vs. hypothetical-projected nominal @ E = {bin_info.energy_mev:.4f} MeV")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n[Q-H1] Does projecting fix the negativity? If yes: the projector works, but production "
      "never invokes it on the nominal — only on MC samples.")

In [ ]:
# --- H2: Per-degree fit grid (L=1..MAX_LEGENDRE_DEGREE) ----------------------
# Force each degree explicitly and inspect chi2/dof, AICc weight, and min(p(μ)) on
# the dense grid so we can see when the curve starts ringing.

mu_dense = np.linspace(-1, 1, 1001)
per_L_results = []
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for L in range(1, MAX_LEGENDRE_DEGREE + 1):
    coef_L_df, info_L = sample_legendre_coefficients(
        exfor_df,
        value_col="value", unc_col="unc",
        degree=L,
        ridge_lambda=RIDGE_LAMBDA, ridge_power=RIDGE_POWER, df_method=DF_METHOD,
        external_weights=kernel_weights,
        n_samples=1,
        rescale_unc_by_chi2=RESCALE_UNC_BY_CHI2,
        allow_shrink_unc=ALLOW_SHRINK_UNC,
        use_band_discrepancy=USE_BAND_DISCREPANCY,
        min_points_per_band=MIN_POINTS_PER_BAND,
        max_band_scale=MAX_BAND_SCALE_FACTOR,
    )
    coefs_L = coef_L_df.iloc[0].to_numpy()
    y_L = legval(mu_dense, coefs_L)
    min_p_L = float(np.min(y_L))
    # AICc is only present when select_degree is requested; with degree=L it is None.
    aicc_L = info_L.get("aicc", np.nan)
    if aicc_L is None:
        aicc_L = np.nan
    per_L_results.append({
        "L": L,
        "chi2_red": info_L["chi2_red"],
        "AICc": float(aicc_L) if aicc_L == aicc_L else np.nan,
        "min_p_dense": min_p_L,
        "negative?": min_p_L < 0,
    })
    ax = axes.flat[L - 1]
    ax.errorbar(exfor_df["mu"], exfor_df["value"], yerr=exfor_df["unc"],
                fmt="o", ms=4, color="k", capsize=2, alpha=0.8)
    ax.plot(mu_dense, y_L, lw=2, color="C0" if min_p_L >= 0 else "C3")
    ax.axhline(0, color="red", ls="--", lw=0.7)
    ax.set_title(f"L={L}  χ²/dof={info_L['chi2_red']:.2f}  min(p)={min_p_L:+.2e}")
    ax.grid(alpha=0.3)
for ax in axes[-1, :]:
    ax.set_xlabel(r"$\mu$")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$d\sigma/d\Omega$ (b/sr)")
plt.suptitle(f"Per-degree fit @ E = {bin_info.energy_mev:.4f} MeV — red curves go negative")
plt.tight_layout()
plt.show()

per_L_df = pd.DataFrame(per_L_results)
print(per_L_df.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("\n[Q-H2] At which L does the curve first go negative? Is L=5 the AICc winner even when "
      "L=3 or L=4 stays positive with similar chi2/dof?")

In [ ]:
# --- H4: Hat-matrix leverage per data point ---------------------------------
# Reproduce H = X (XᵀWX + λΛ)⁻¹ XᵀW from _weighted_ridge_fit so we can see which
# points control the fit. High diag(H) = high leverage.

L = fitted_degree
sigma = exfor_df["unc"].values
mu = exfor_df["mu"].values
y = exfor_df["value"].values

w = (kernel_weights / sigma**2)
X = legvander(mu, L)  # (n, L+1)
W = np.diag(w)

# Ridge penalty matrix: Λ_ll = l^ridge_power for l>=1
Lam = np.diag([0.0] + [l ** RIDGE_POWER for l in range(1, L + 1)])
XtWX = X.T @ W @ X + RIDGE_LAMBDA * Lam
H = X @ np.linalg.solve(XtWX, X.T @ W)
lev = np.diag(H)

lev_df = pd.DataFrame({
    "experiment_id": exfor_df["experiment_id"].values,
    "mu": mu,
    "value": y,
    "unc": sigma,
    "kernel_weight": kernel_weights,
    "leverage": lev,
})
print(lev_df.sort_values("leverage", ascending=False).to_string(index=False,
      float_format=lambda x: f"{x:.4g}"))
print(f"\nsum(diag(H)) = {lev.sum():.3f}  (≈ effective # parameters)")
print()
per_exp = lev_df.groupby("experiment_id")["leverage"].sum().sort_values(ascending=False)
print("Total leverage per experiment:")
print(per_exp.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
colors = {eid: f"C{i}" for i, eid in enumerate(lev_df["experiment_id"].unique())}
ax.bar(np.arange(len(lev_df)), lev,
       color=[colors[e] for e in lev_df["experiment_id"]])
for eid, c in colors.items():
    ax.bar([], [], color=c, label=eid)
for j, m in enumerate(mu):
    if m < -0.5:
        ax.axvspan(j-0.5, j+0.5, alpha=0.07, color="red")
    elif m > 0.5:
        ax.axvspan(j-0.5, j+0.5, alpha=0.07, color="blue")
ax.set_xticks(np.arange(len(lev_df)))
ax.set_xticklabels([f"μ={m:+.2f}" for m in mu], rotation=60, fontsize=8)
ax.set_ylabel("diag(H)  (leverage)")
ax.set_title(f"Per-point leverage @ E = {bin_info.energy_mev:.4f} MeV (L={L})")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("\n[Q-H4] Do Cierjacks BACKWARD points carry disproportionate leverage despite the floored σ? "
      "Does total Cierjacks leverage exceed Kinney's despite Kinney having more points?")

## What-if variants

Each variant changes ONE knob (or, in the case of `floor+cap` and `per_band_weight_cap`, a
small targeted combination) and re-runs filter + nominal fit. We compare:
- the dσ/dΩ curve (does it go negative?)
- chi²/dof
- min(p(μ)) on the dense grid
- the chosen / kept degrees
- per-experiment weight totals

This is exploratory — no production code is changed. The goal is to isolate which knob
recovers a physical backward shape and at what cost in the forward/mid bands.

In [ ]:
# --- run_variant + notebook-local per-band weight cap ------------------------

# Band thresholds match scripts/resample_AD.py:146
def _band_of(mu):
    if mu > 0.5:
        return "F"
    if mu < -0.5:
        return "B"
    return "M"


def _per_band_weight_cap(exfor_df, kernel_weights, max_frac=MAX_EXP_WEIGHT_FRAC_BIN):
    # Apply MAX_EXP_WEIGHT_FRAC_BIN separately within each μ band {F, M, B}.
    # Inside each band, reuse `apply_per_experiment_weight_cap` on the slice.
    # This is a notebook-local experiment, NOT a production change.
    bands = np.array([_band_of(m) for m in exfor_df["mu"].values])
    out = kernel_weights.copy()
    for b in ("F", "M", "B"):
        idx = np.where(bands == b)[0]
        if len(idx) == 0:
            continue
        sub = exfor_df.iloc[idx].reset_index(drop=True)
        kw_sub = kernel_weights[idx].copy()
        if len(sub["entry"].unique()) <= 1:
            continue  # only one experiment in this band — nothing to cap
        capped, _diag, _applied = apply_per_experiment_weight_cap(
            sub, kw_sub, max_experiment_weight_fraction=max_frac,
        )
        out[idx] = capped
    return out


def run_variant(name, **overrides):
    # Re-execute filter + fit with config overrides. Returns dict of diagnostics.
    cfg = dict(
        min_relative_uncertainty=MIN_RELATIVE_UNCERTAINTY,
        unc_floor_strategy=UNCERTAINTY_FLOOR_STRATEGY,
        normalize_by_n_points=NORMALIZE_BY_N_POINTS,
        sigma_norm=NORMALIZATION_SIGMA,
        max_experiment_weight_fraction=MAX_EXP_WEIGHT_FRAC_BIN,
        exclude_experiments=list(EXCLUDE_EXPERIMENTS),
        max_degree=MAX_LEGENDRE_DEGREE,
    )
    project_nominal = overrides.pop("project_nominal", False)
    drop_experiments = overrides.pop("drop_experiments", [])
    use_per_band_cap = overrides.pop("use_per_band_cap", False)
    if drop_experiments:
        cfg["exclude_experiments"] = list(set(cfg["exclude_experiments"] + drop_experiments))
    cfg.update(overrides)

    df_v, exps_v, kw_v, diag_v, _floor_v = filter_exfor_with_energy_bin(
        exfor_cache=exfor_cache,
        sorted_energies=sorted_exfor_energies,
        bin_lower_mev=bin_info.bin_lower_mev,
        bin_upper_mev=bin_info.bin_upper_mev,
        target_energy_mev=bin_info.energy_mev,
        m_proj_u=M_PROJ_U,
        m_targ_u=M_TARG_U,
        dedupe_per_experiment=True,
        exclude_experiments=cfg["exclude_experiments"],
        min_relative_uncertainty=cfg["min_relative_uncertainty"],
        unc_floor_strategy=cfg["unc_floor_strategy"],
        normalize_by_n_points=cfg["normalize_by_n_points"],
        sigma_norm=cfg["sigma_norm"],
        max_experiment_weight_fraction=cfg["max_experiment_weight_fraction"],
    )
    if "experiment_id" not in df_v.columns:
        df_v["experiment_id"] = df_v["entry"].astype(str) + "/" + df_v["subentry"].astype(str)
    if use_per_band_cap:
        kw_v = _per_band_weight_cap(df_v, kw_v, max_frac=cfg["max_experiment_weight_fraction"])

    coef_v, fit_v = sample_legendre_coefficients(
        df_v, value_col="value", unc_col="unc",
        degree=overrides.get("force_degree", None),
        max_degree=cfg["max_degree"],
        select_degree=SELECT_DEGREE,
        ridge_lambda=RIDGE_LAMBDA, ridge_power=RIDGE_POWER, df_method=DF_METHOD,
        external_weights=kw_v, n_samples=1,
        rescale_unc_by_chi2=RESCALE_UNC_BY_CHI2,
        allow_shrink_unc=ALLOW_SHRINK_UNC,
        use_band_discrepancy=USE_BAND_DISCREPANCY,
        min_points_per_band=MIN_POINTS_PER_BAND,
        max_band_scale=MAX_BAND_SCALE_FACTOR,
    )
    coefs = coef_v.iloc[0].to_numpy()
    if project_nominal:
        coefs = project_to_positive_distribution(
            coefs, n_points=POSITIVITY_CHECK_POINTS, frozen_indices={0: float(coefs[0])},
        )

    mu_dense_v = np.linspace(-1, 1, 1001)
    y_dense_v = legval(mu_dense_v, coefs)
    return {
        "name": name,
        "coeffs": coefs,
        "df": df_v,
        "kw": kw_v,
        "fit_info": fit_v,
        "chi2_red": float(fit_v["chi2_red"]),
        "degree": int(fit_v["degree"]),
        "min_p_dense": float(np.min(y_dense_v)),
        "y_dense": y_dense_v,
        "n_eff": float(diag_v.n_eff),
    }


# Smoke-test the helper
_test = run_variant("baseline_test")
print(f"baseline_test: L={_test['degree']}, chi2/dof={_test['chi2_red']:.3f}, "
      f"min(p)={_test['min_p_dense']:+.3e}")

In [ ]:
# --- Run all what-if variants ------------------------------------------------
variants = [
    ("baseline", {}),
    ("positivity_on_nominal", {"project_nominal": True}),
    ("cap_L_at_4", {"max_degree": 4}),
    ("cap_L_at_3", {"max_degree": 3}),
    ("tighter_exp_cap", {"max_experiment_weight_fraction": 0.5}),
    ("disable_n_points", {"normalize_by_n_points": False}),
    ("no_unc_floor", {"min_relative_uncertainty": 0.0}),
    ("floor_plus_tighter_cap", {"max_experiment_weight_fraction": 0.5,
                                 "min_relative_uncertainty": 0.0}),
    ("drop_cierjacks", {"drop_experiments": ["20743"]}),
    ("per_band_weight_cap", {"use_per_band_cap": True}),
]
results = [run_variant(n, **kw) for n, kw in variants]

# Build summary
summary = pd.DataFrame([
    {
        "variant": r["name"],
        "L": r["degree"],
        "chi2/dof": r["chi2_red"],
        "min(p) dense": r["min_p_dense"],
        "negative?": r["min_p_dense"] < 0,
        "N_eff": r["n_eff"],
        "n_pts": len(r["df"]),
    }
    for r in results
])
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

In [ ]:
# --- Plot all variant curves on data ----------------------------------------
mu_dense = np.linspace(-1, 1, 1001)
fig, ax = plt.subplots(figsize=(11, 7))
ax.errorbar(exfor_df["mu"], exfor_df["value"], yerr=exfor_df["unc"],
            fmt="o", color="k", capsize=3, ms=5, label="EXFOR (post-floor)", zorder=10)
for r in results:
    style = "--" if r["min_p_dense"] >= 0 else "-"
    ax.plot(mu_dense, r["y_dense"], style, lw=1.5,
            label=f"{r['name']}  L={r['degree']}  χ²/dof={r['chi2_red']:.2f}  min={r['min_p_dense']:+.1e}")
ax.axhline(0, color="red", ls=":", lw=1)
ax.axvspan(-1, -0.5, alpha=0.07, color="red")
ax.axvspan(0.5, 1, alpha=0.07, color="blue")
ax.set_xlabel(r"$\mu$")
ax.set_ylabel(r"$d\sigma/d\Omega$ (b/sr)")
ax.set_title(f"What-if variants @ E = {bin_info.energy_mev:.4f} MeV  "
             "(solid = goes negative; dashed = stays positive)")
ax.legend(fontsize=8, loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Per-band weight totals for the per_band_weight_cap variant vs baseline
def per_band_weight_table(df_v, kw_v):
    bands = pd.Series([_band_of(m) for m in df_v["mu"].values], index=df_v.index, name="band")
    return (
        pd.DataFrame({"experiment_id": df_v["experiment_id"].values, "w": kw_v, "band": bands.values})
        .pivot_table(index="experiment_id", columns="band", values="w", aggfunc="sum", fill_value=0.0)
        .pipe(lambda x: x / x.sum().sum())
    )

baseline_r = next(r for r in results if r["name"] == "baseline")
band_r = next(r for r in results if r["name"] == "per_band_weight_cap")
print("Per-band weight fractions — BASELINE")
print(per_band_weight_table(baseline_r["df"], baseline_r["kw"]).to_string(
    float_format=lambda x: f"{x:.3f}"))
print()
print("Per-band weight fractions — per_band_weight_cap")
print(per_band_weight_table(band_r["df"], band_r["kw"]).to_string(
    float_format=lambda x: f"{x:.3f}"))

## Findings (fill in after running all cells)

**Hypothesis status** — mark each PASS / FAIL / PARTIAL with the cell that proved it.

- **H1 — Positivity projection only on samples, never nominal.**
  - `check_angular_distribution_positivity` on nominal: ___ (Cell 8)
  - `min(p)` BEFORE projection: ___ (Cell 8)
  - `min(p)` AFTER hypothetical projection: ___ (Cell 8)
  - Status: ___

- **H2 — L=5 with N_eff=2.8 rings.**
  - AICc winner: ___, AICc weight on L=5: ___ (Cell 7)
  - First L where `min(p) < 0`: ___ (Cell 9)
  - Status: ___

- **H3 — Asymmetric uncertainty floor on Cierjacks.**
  - n_floored on Cierjacks backward: ___ / ___ (Cell 5)
  - rel_raw mean (Cierjacks backward): ___, rel_floored: ___ (Cell 5)
  - Status: ___

- **H4 — GLS-ESS / cap mechanics.**
  - Kinney shrink factor median: ___, Cierjacks shrink factor median: ___ (Cell 6)
  - Total leverage Cierjacks vs Kinney: ___ vs ___ (Cell 10)
  - Status: ___

**Variant outcomes** (fill from Cell 13/14 summary):

| variant | L | χ²/dof | min(p) | comment |
|---|---|---|---|---|
| baseline | _ | _ | _ |  |
| positivity_on_nominal | _ | _ | _ |  |
| cap_L_at_4 | _ | _ | _ |  |
| tighter_exp_cap (0.5) | _ | _ | _ |  |
| per_band_weight_cap | _ | _ | _ |  |
| drop_cierjacks | _ | _ | _ |  |

**Recommended fix** (write after running):

> _e.g., "Apply project_to_positive_distribution to the nominal coeffs alongside MC samples
> at scripts/exfor_to_endf_sampling_v2.py:1205, AND lower MAX_LEGENDRE_DEGREE to 4 for bins
> with N_eff < 5"._